# BC v2 Training on Colab A100

Trains the ObjectCentricPolicy on the bc_v2 corpus (12,425 episodes, 4.41 GB gz) collected via `scripts/collect_gt_warmstart.py`.

**Plan:** see `.claude/doc/bc_v2_training_plan.md` (Path C, A100 cloud).

**Required runtime:** A100 40 GB. Notebook **aborts** if a different GPU is allocated (don't waste compute units on a T4).

**Required Drive upload (do this BEFORE starting the notebook):**
- The data file at `/content/drive/MyDrive/ARC2026_AGI_3/Input_Data/episodes_bc_v2.jsonl.gz` (4.41 GB).

**Project code is cloned from GitHub** in Cell 4 — no Drive upload needed. Defaults to repo `https://github.com/QAQWillQwQ/ARC-Prize-2026-ARC-AGI-3.git` branch `jihang`.

**Compute unit budget (200 units, A100 40 GB ≈ 11.77 units/h):**
- Smoke run (1 epoch on tiny subset): ~5–10 units
- Full bc_v2 training (16 epochs): ~100–110 units
- Reserve: ~80 units for one retry / iteration

Approx wall: **6–9 h continuous** for the full run. Colab session cap is 12 h, so a single uninterrupted session covers it. Checkpoints copy back to Drive every epoch in case of disconnect.

## 1. Verify GPU is A100 (abort if not)

In [ ]:
import subprocess
import sys

out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('Allocated GPU:', out)

if 'A100' not in out:
    raise SystemExit(
        f'ABORT: expected A100, got: {out!r}\n'
        f'Go to Runtime > Change runtime type and pick A100.\n'
        f'Do not run on a smaller GPU (profile a100 needs >12 GB VRAM and the wall on T4 / L4 will exceed your unit budget).'
    )
print('OK: A100 confirmed.')

## 2. Mount Google Drive

Required for reading the data .gz and writing checkpoints back so they survive session disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure paths and run name

Edit the values below to match your setup. Defaults clone the `jihang` branch.

In [ ]:
from pathlib import Path
from datetime import datetime

# ===== EDIT THESE =====
GITHUB_REPO_URL    = 'https://github.com/QAQWillQwQ/ARC-Prize-2026-ARC-AGI-3.git'
GITHUB_BRANCH      = 'jihang'   # the user's working branch
DRIVE_INPUT_GZ     = Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data/episodes_bc_v2.jsonl.gz')
RUN_NAME           = 'bc_v2_baseline_a100'
EPOCHS_OVERRIDE    = None      # None -> use profile's epochs=16; else int
# ===== END EDIT =====

DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Training_Output')
LOCAL_WORKDIR     = Path('/content/ARC-Prize-2026-ARC-AGI-3')
LOCAL_INPUT_DIR   = Path('/content/bc_v2_input')
LOCAL_OUTPUT_DIR  = Path('/content/bc_v2_output')

RUN_TS         = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
FULL_RUN_NAME  = f'{RUN_NAME}_{RUN_TS}'
DRIVE_OUTPUT   = DRIVE_OUTPUT_BASE / FULL_RUN_NAME
LOCAL_OUTPUT   = LOCAL_OUTPUT_DIR / FULL_RUN_NAME

for p in [DRIVE_OUTPUT_BASE, LOCAL_INPUT_DIR, LOCAL_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

print('github repo      :', GITHUB_REPO_URL)
print('github branch    :', GITHUB_BRANCH)
print('drive input gz   :', DRIVE_INPUT_GZ, '(exists:', DRIVE_INPUT_GZ.exists(), ')')
print('drive output dir :', DRIVE_OUTPUT)
print('local workdir    :', LOCAL_WORKDIR)
print('local output dir :', LOCAL_OUTPUT)

if not DRIVE_INPUT_GZ.exists():
    raise SystemExit(f'Drive input .gz not found: {DRIVE_INPUT_GZ}. Upload the data first.')

## 4. Clone the project from GitHub (and switch to the working branch)

Clones into `/content/ARC-Prize-2026-ARC-AGI-3` and checks out `GITHUB_BRANCH` (default `jihang`).

If the directory already exists from a previous run, it's wiped and re-cloned to ensure a fresh, branch-correct copy.

**Private repo?** If `git clone` prompts for credentials or fails with 403/404, the repo is private. Two options:
1. Make the repo public, OR
2. Use a Personal Access Token: replace `GITHUB_REPO_URL` with `https://<USERNAME>:<TOKEN>@github.com/QAQWillQwQ/ARC-Prize-2026-ARC-AGI-3.git`.

In [ ]:
import shutil
import subprocess

if LOCAL_WORKDIR.exists():
    print(f'removing existing {LOCAL_WORKDIR} for a clean clone')
    shutil.rmtree(LOCAL_WORKDIR)

# Shallow clone: --depth 1 fetches only the latest commit on the branch.
# Saves time and disk vs full history; we don't need git log here.
print(f'git clone --depth 1 --branch {GITHUB_BRANCH} {GITHUB_REPO_URL} {LOCAL_WORKDIR}')
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(LOCAL_WORKDIR)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise SystemExit(f'git clone failed (exit {result.returncode}). See stderr above.')

# Confirm we landed on the right branch + show the head commit
head = subprocess.run(['git', '-C', str(LOCAL_WORKDIR), 'log', '-1', '--format=%H %s'],
                      capture_output=True, text=True).stdout.strip()
branch = subprocess.run(['git', '-C', str(LOCAL_WORKDIR), 'rev-parse', '--abbrev-ref', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
print(f'\nhead commit: {head}')
print(f'branch     : {branch}')
if branch != GITHUB_BRANCH:
    raise SystemExit(f'expected branch {GITHUB_BRANCH}, got {branch}')

# Verify the critical paths
print()
print('verify project layout:')
for sub in ['src', 'scripts', 'arc_agi_3_wheels', 'environment_files']:
    p = LOCAL_WORKDIR / sub
    print(f'  {sub}: {p.exists()} ({len(list(p.glob("*"))) if p.exists() else 0} entries)')

## 5. Copy data .gz from Drive to local SSD

Reading 4.4 GB through the dataloader directly from Drive is slow. Copying once front-loads the I/O cost into one ~3–5 min copy.

In [ ]:
LOCAL_INPUT_GZ = LOCAL_INPUT_DIR / 'episodes_bc_v2.jsonl.gz'
if LOCAL_INPUT_GZ.exists():
    print(f'already present: {LOCAL_INPUT_GZ} ({LOCAL_INPUT_GZ.stat().st_size / 1e9:.2f} GB)')
else:
    print(f'copying {DRIVE_INPUT_GZ} -> {LOCAL_INPUT_GZ} ...')
    shutil.copy2(DRIVE_INPUT_GZ, LOCAL_INPUT_GZ)
    print(f'done: {LOCAL_INPUT_GZ.stat().st_size / 1e9:.2f} GB')

## 6. Install bundled wheels

`arc_agi` and `arcengine` are not on PyPI for Linux. The wheels in `arc_agi_3_wheels/` are manylinux cp312 — they install cleanly on Colab Linux.

In [ ]:
wheels_dir = LOCAL_WORKDIR / 'arc_agi_3_wheels'
wheels = sorted(wheels_dir.glob('*.whl'))
print(f'installing {len(wheels)} wheels from {wheels_dir}')
!pip install -q {wheels_dir}/*.whl

!pip install -q py-spy

import importlib
for mod in ['arc_agi', 'arcengine', 'torch']:
    m = importlib.import_module(mod)
    print(f'  {mod}: {getattr(m, "__version__", "?")}  ({m.__file__})')

import torch
print(f'torch.cuda.is_available: {torch.cuda.is_available()}')
print(f'torch.cuda.get_device_name: {torch.cuda.get_device_name(0)}')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')

## 7. Verify the data file end-to-end

Before burning compute on training, confirm the .gz is readable to its last episode (catches upload corruption / partial transfers).

In [ ]:
import gzip
import json
import time
from collections import Counter

t0 = time.time()
n = 0
buckets = Counter()
games = Counter()
states = Counter()

try:
    with gzip.open(LOCAL_INPUT_GZ, 'rt') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ep = json.loads(line)
            n += 1
            buckets[ep.get('source', '?')] += 1
            games[ep.get('short_id', '?')] += 1
            states[ep.get('final_state', '?')] += 1
    elapsed = time.time() - t0
    print(f'OK: {n:,} episodes read in {elapsed:.1f}s')
    print(f'  buckets: {dict(buckets)}')
    print(f'  games:   {len(games)} ({sorted(games)})')
    print(f'  states:  {dict(states)}')
except Exception as e:
    raise SystemExit(f'FAIL: read error after {n} episodes: {type(e).__name__}: {e}')

## 8. (Optional) Smoke run: 1 epoch on a single game

Burns ~3–5 units to verify the training pipeline runs end-to-end before committing to the long run.

Skip this if you've already validated the pipeline. Set `RUN_SMOKE = False` to skip.

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    smoke_out = LOCAL_OUTPUT_DIR / f'smoke_{RUN_TS}'
    smoke_out.mkdir(parents=True, exist_ok=True)
    print(f'smoke run -> {smoke_out}')
    !cd {LOCAL_WORKDIR} && python -m src.train \
        --project-root {LOCAL_WORKDIR} \
        --data {LOCAL_INPUT_GZ} \
        --output-dir {smoke_out} \
        --hardware-profile a100 \
        --games sp80 \
        --epochs 1 \
        --online-val-every 1000
    print('--- smoke files written ---')
    !ls -la {smoke_out}/checkpoints/ 2>/dev/null
else:
    print('skipping smoke (RUN_SMOKE=False)')

## 9. Full training run

Uses the `a100` hardware profile (batch=192, model_dim=384, depth=6, slots=8, history=4, epochs=16, online_val_every=2). Output streams live so you can watch loss / val score.

**Caveat:** this notebook uses the **existing** `train.py` flow. The `bc_v2_3stage` curriculum + bucket weighting + phase mask described in §4–§6 of `bc_v2_training_plan.md` are NOT yet implemented in train.py — this run is a single-stage baseline across all buckets at equal weight. It establishes a baseline; the full plan's curriculum will be a follow-up iteration.

**Wall estimate:** 6–9 h. Stay on the page or rejoin the runtime occasionally — Colab disconnects idle browsers eventually.

In [ ]:
epochs_arg = f'--epochs {EPOCHS_OVERRIDE}' if EPOCHS_OVERRIDE else ''

print(f'run name : {FULL_RUN_NAME}')
print(f'output   : {LOCAL_OUTPUT}')
print(f'will copy back to: {DRIVE_OUTPUT}')
print()

!cd {LOCAL_WORKDIR} && python -m src.train \
    --project-root {LOCAL_WORKDIR} \
    --data {LOCAL_INPUT_GZ} \
    --output-dir {LOCAL_OUTPUT} \
    --hardware-profile a100 \
    {epochs_arg} \
    --online-val-every 2

## 10. Copy results back to Drive

On graceful completion this saves the full output dir to Drive. If the cell above gets interrupted (Colab disconnect, KeyboardInterrupt), run THIS cell anyway — it will copy whatever made it to local disk so far (last.pth + interrupt.pth + any best.pth).

In [ ]:
print(f'copying {LOCAL_OUTPUT} -> {DRIVE_OUTPUT}')
!rsync -a --info=progress2 "{LOCAL_OUTPUT}/" "{DRIVE_OUTPUT}/"
print()
print('files on Drive:')
!ls -la {DRIVE_OUTPUT}
!ls -la {DRIVE_OUTPUT}/checkpoints 2>/dev/null

## 11. (Optional) Periodic checkpoint sync

If you want continuous protection against disconnect (vs only the final copy in §10), run this cell **in a parallel browser tab** while §9 trains. It rsyncs the local output to Drive every 5 min. Stop it manually after §9 finishes.

In [ ]:
import time
import shutil
import subprocess

while True:
    if LOCAL_OUTPUT.exists():
        subprocess.run(['rsync', '-a', f'{LOCAL_OUTPUT}/', f'{DRIVE_OUTPUT}/'], check=False)
        print(f'[{time.strftime("%H:%M:%S")}] synced {LOCAL_OUTPUT} -> {DRIVE_OUTPUT}')
    else:
        print(f'[{time.strftime("%H:%M:%S")}] no local output yet')
    time.sleep(300)

## 12. Resuming after a disconnect

If Colab kicked you off mid-training, restart the runtime, re-run cells 1–7 (everything before training), then run this cell **instead of** §9. It picks up from `last.pth` (or `interrupt.pth` if a clean Ctrl-C) on Drive.

In [ ]:
prior_runs = sorted(DRIVE_OUTPUT_BASE.glob(f'{RUN_NAME}_*'), reverse=True)
print(f'found {len(prior_runs)} prior runs matching {RUN_NAME}_*')
for p in prior_runs[:5]:
    print(f'  {p.name}')

if not prior_runs:
    raise SystemExit('No prior runs found to resume from.')

RESUME_FROM_DRIVE = prior_runs[0]
ckpt_dir = RESUME_FROM_DRIVE / 'checkpoints'
candidates = ['interrupt.pth', 'last.pth', 'best.pth']
resume_ckpt = next((ckpt_dir / c for c in candidates if (ckpt_dir / c).exists()), None)
if resume_ckpt is None:
    raise SystemExit(f'No checkpoint found in {ckpt_dir}')
print(f'resuming from: {resume_ckpt}')

if not LOCAL_OUTPUT.exists():
    LOCAL_OUTPUT.mkdir(parents=True)
!rsync -a "{RESUME_FROM_DRIVE}/" "{LOCAL_OUTPUT}/"
local_ckpt = LOCAL_OUTPUT / 'checkpoints' / resume_ckpt.name

epochs_arg = f'--epochs {EPOCHS_OVERRIDE}' if EPOCHS_OVERRIDE else ''
!cd {LOCAL_WORKDIR} && python -m src.train \
    --project-root {LOCAL_WORKDIR} \
    --data {LOCAL_INPUT_GZ} \
    --output-dir {LOCAL_OUTPUT} \
    --hardware-profile a100 \
    {epochs_arg} \
    --online-val-every 2 \
    --resume {local_ckpt}